In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(font_scale=2.2,style='white')
import pandas as pd
import numpy as np
import scipy
import pickle as pkl
from collections import OrderedDict 


In [2]:
# goal: to figure out how many severe symptoms were occurring in
# participants who had severe reactions' normal life

In [3]:

with open('v3_participant_nums.pkl', 'rb') as f:
    v3_participant_nums = pkl.load(f)

In [4]:

latest_filename = '/data/permed/unified/dataframe/20210923/df_q_unified_pilot12_20210923.csv'
df = pd.read_csv(latest_filename, low_memory=False)

In [5]:
def get_heat_over_389_participants(df):
    test_df = df[pd.to_numeric(df.quest_heat, errors='coerce').notnull()]
    test_df['quest_heat_numeric'] = pd.to_numeric(test_df['quest_heat'])
    heat_over_389_rows = test_df.where(test_df['quest_heat_numeric'] > 38.9).where(test_df['quest_heat_numeric'] < 41).dropna(how='all')
    return list(heat_over_389_rows.index)


In [6]:
heat_over_389_participants = get_heat_over_389_participants(df)

for part in heat_over_389_participants:
    df.loc[part, "quest_symptoms"] = df.loc[part, "quest_symptoms"] + ',heat over 38.9'

<ipython-input-5-f77aac9f8aab>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['quest_heat_numeric'] = pd.to_numeric(test_df['quest_heat'])


In [7]:
severe_participants = df[df['participant_num'].isin(v3_participant_nums[0])]

In [8]:
severe_participants_baseline = severe_participants[((severe_participants['days_from_q_to_first'] <=-7)
                                                   | (severe_participants['days_from_q_to_first'] >14))
                                                   & ((severe_participants['days_from_q_to_second'] <=-7)
                                                   | (severe_participants['days_from_q_to_second'] >14))
                                                   & ((severe_participants['days_from_q_to_third'] <=-7)
                                                   | (severe_participants['days_from_q_to_third'] >14))]

In [9]:
severe_participants_baseline[['participant_num', 'quest_symptoms']]

,participant_num,quest_symptoms
15434,1130,healthy
15435,1130,healthy
15436,1130,healthy
15437,1130,healthy
15438,1130,healthy
...,...,...
108059,832,healthy
108060,832,healthy
108061,832,healthy
108062,832,healthy


In [10]:
severe_participants_baseline["quest_symptoms_processed"] = severe_participants_baseline["quest_symptoms"].replace({'-1': '',
                                                                             'skipped_question': '',
                                                                             np.nan: ''}, regex=True)

<ipython-input-10-901ff61a6c60>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  severe_participants_baseline["quest_symptoms_processed"] = severe_participants_baseline["quest_symptoms"].replace({'-1': '',


In [11]:
# Add helper methods and objects

#First, categorize heat and fever into > 38.9 and <= 38.9
#Severe: ['chest_pain', 'dyspnea', 'heat over 38.9', 'confusion', 'chills']
 
#Mild: everything else: ['abdominal_pain', 'feel_heat', 'heat_over_37.5', 'back_or_neck_pain', 'cold', 'muscles_pain', 'weakness', 'headache',
#'dizziness', 'vomiting', 'sore_throat', 'diarrhea', 'cough', 'hand_muscles_pain', 'leg_pain', 'ear_pain', 'taste_smell', 'lymph',
# 'fast_heartbeat', 'hypertension']


symp_titles = {'healthy':'No\n Reaction',
               'chest_pain': 'Chest\n Pain',
               'dyspnea':'Dyspnea',
               'heat over 38.9':'Fever\n >38.9',
               'confusion':'Confusion',
               'chills':'Chills',
               'abdominal_pain':'Muscle\n Pain',
               'feel_heat':'Fever\n <=38.9',
               'heat over 37.5':'Fever\n <=38.9',
               'back_or_neck_pain':'Muscle\n Pain',
               'cold':'Cold',
               'muscles_pain':'Muscle\n Pain',
               'weakness':'Fatigue',
               'headache':'Headache',
               'dizziness':'Dizziness',
               'vomiting':'Vomiting',
               'sore_throat':'Sore\n Throat',
               'diarrhea':'Diarrhea',
               'cough':'Cough',
               'hand_muscles_pain':'Muscle\n Pain',
               'leg_pain':'Muscle\n Pain',
               'ear_pain':'Muscle\n Pain',
               'taste_smell':'Loss of\nTaste/Smell',
               'lymph':'Lymph',
               'fast_heartbeat':'Fast Heartbeat',
               'hypertension':'Hypertension',
               'other':'Other',
              '' : 'Skipped Questionnaire'}

possible_symptoms = symp_titles.keys()

def classify_heat(heat_str):
    heat_array = heat_str.split(':')
    if heat_array[0] != 'heat':
        return ""
    try:
        temp = float(heat_array[1])
        # One person made a typo, heat is 636.6
        if temp >= 300:
            temp = temp - 600
        # A couple people did not report 30 with their heat, e.g. reporting 6.4 instead of 36.4
        if temp <= 10:
            temp = temp + 30
            
        if temp <= 30 or temp >= 41:
            return "error with heat %.2f" % (temp)
        
        if temp > 38.9:
            return "heat over 38.9"
        if temp > 37.5:
            return "heat over 37.5"
        else:# temp >= 35:
            return "no heat"
    # If the heat could not be converted to float, assume person is healthy
    except:
        return "no heat"

def preprocess_symptom_array(arr):
    arr = arr.copy()
    for i in range(len(arr)):
        s = arr[i]
        if "heat:" in s:
            arr[i] = classify_heat(s)
    return list(set(arr))

In [12]:
# Assert that all symptoms are accounted for in the symptoms listed
# in the above dictionary or by processing their heat entry

severe_symptoms = ['chest_pain', 'dyspnea', 'heat over 38.9', 'confusion', 'chills']
count_severe = 0

for i,r in severe_participants_baseline.iterrows():
    symptoms = r["quest_symptoms_processed"].split(',')

    has_severe = False
    for symptom in symptoms:
        #print(symptoms)
        assert((symptom in possible_symptoms)
               or (classify_heat(symptom) in possible_symptoms)
               or (classify_heat(symptom) == 'no heat'))
        if symptom in severe_symptoms or classify_heat(symptom) in severe_symptoms:
            has_severe = True

    if has_severe:
        count_severe += 1

In [13]:
count_severe

50

In [14]:
def beta(value,n):
    if value == 0:
        return [0,0]
    lower = value-scipy.stats.beta.ppf(0.025, value, n-value, loc=0, scale=1)*n
    upper = scipy.stats.beta.ppf(0.975, value, n-value, loc=0, scale=1)*n-value
    if lower > value:
        lower = value
    return [value - lower, value + upper]

In [15]:
a, b = beta(count_severe, len(severe_participants_baseline))

In [16]:
num_sev_parts = len(severe_participants_baseline)

In [19]:
print(a)
print(b)
print(num_sev_parts)

37.22532146971909
64.58312811230472
2270


In [17]:
a / num_sev_parts

0.016398820030713256

In [18]:
b / num_sev_parts

0.028450717230090187